# OULAD Week-4 Early Failure Prediction

This notebook reconstructs the week-4 feature table, fits the two classification pipelines, and summarises the reported week-4 results.

## 1. Run The Analysis Workflow

Running the following cell rebuilds the week-4 features, refits the logistic regression and CatBoost pipelines, and refreshes the main output artifacts. The raw OULAD CSV files are not stored in this public repository, so a local rerun requires those files in the project root.

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

from week4_pipeline import run_full_pipeline

project_root = Path.cwd()
results = run_full_pipeline(project_root, force_rebuild=False)

display(Markdown(f"Feature table: `{results['feature_table_path']}`"))
display(Markdown(f"Metrics summary: `{results['metrics_summary_path']}`"))
display(Markdown(f"Threshold sweep: `{results['threshold_sweep_path']}`"))

Feature table: `C:\Users\giftv\Documents\4th IE and shared year courses\ELEN4025\Pipeline 2\artifacts\week4_feature_table.parquet`

Metrics summary: `C:\Users\giftv\Documents\4th IE and shared year courses\ELEN4025\Pipeline 2\artifacts\metrics_summary.csv`

Threshold sweep: `C:\Users\giftv\Documents\4th IE and shared year courses\ELEN4025\Pipeline 2\artifacts\threshold_sweep.csv`

## 2. Final Test-Set Comparison

In [2]:
results['comparison_table']

,model,split,threshold,accuracy,log_loss,precision,recall,specificity,fpr,f1,roc_auc,tn,fp,fn,tp
0,Logistic Regression,test,0.5,0.737383,0.509206,0.701744,0.771531,0.706856,0.293144,0.734985,0.818487,2433,1009,703,2374
1,Logistic Regression (tuned threshold),test,0.3,0.708391,0.509206,0.627328,0.941501,0.500000,0.500000,0.752956,0.818487,1721,1721,180,2897
2,CatBoost,test,0.5,0.741371,0.498947,0.696303,0.801755,0.687391,0.312609,0.745317,0.826431,2366,1076,610,2467
3,Calibrated CatBoost (tuned threshold),test,0.3,0.729406,0.504381,0.652144,0.914527,0.563916,0.436084,0.761364,0.826431,1941,1501,263,2814


## 3. Validation Threshold Sweep

In [3]:
results['threshold_sweep'].query("split == 'validation'")

,model,split,threshold,accuracy,log_loss,precision,recall,specificity,fpr,f1,roc_auc,tn,fp,fn,tp
0,Logistic Regression,validation,0.1,0.656082,0.49563,0.579448,0.989600,0.357931,0.642069,0.730917,0.827442,1232,2210,32,3045
1,Calibrated CatBoost,validation,0.1,0.656389,0.49208,0.579427,0.992200,0.356188,0.643812,0.731608,0.835842,1226,2216,24,3053
2,Logistic Regression,validation,0.2,0.686302,0.49563,0.603200,0.980175,0.423591,0.576409,0.746812,0.827442,1458,1984,61,3016
3,Calibrated CatBoost,validation,0.2,0.706703,0.49208,0.621684,0.967176,0.473852,0.526148,0.756867,0.835842,1631,1811,101,2976
4,Logistic Regression,validation,0.3,0.714067,0.49563,0.630458,0.952551,0.500872,0.499128,0.758737,0.827442,1724,1718,146,2931
5,Calibrated CatBoost,validation,0.3,0.732628,0.49208,0.653616,0.922327,0.563045,0.436955,0.765063,0.835842,1938,1504,239,2838
6,Logistic Regression,validation,0.4,0.732628,0.49563,0.661815,0.886578,0.595003,0.404997,0.757883,0.827442,2048,1394,349,2728
7,Calibrated CatBoost,validation,0.4,0.747967,0.49208,0.684889,0.863178,0.644974,0.355026,0.763767,0.835842,2220,1222,421,2656
8,Logistic Regression,validation,0.5,0.741525,0.49563,0.706161,0.774781,0.711795,0.288205,0.738881,0.827442,2450,992,693,2384
9,Calibrated CatBoost,validation,0.5,0.749041,0.49208,0.712600,0.784855,0.717025,0.282975,0.746984,0.835842,2468,974,662,2415


## 4. Pipeline Comparison Table

In [4]:
pd.read_csv(project_root / 'artifacts' / 'submission' / 'pipeline_comparison.csv')

,Pipeline ID,Feature set,Encoding,Scaling,Model,Evaluation method,Hyperparameters,Seed,Chosen threshold,Accuracy,Precision,Recall,Specificity,FPR,F1,Log loss,AUC
0,P1,All week-4 features,Most-frequent imputation + one-hot categorical...,StandardScaler on numeric columns only,LogisticRegression(max_iter=1000),Split 60/20/20 with threshold selection on val...,"max_iter=1000, random_state=42",42,0.5,0.7374,0.7017,0.7715,0.7069,0.2931,0.7350,0.5092,0.8185
1,P1-T,All week-4 features,Most-frequent imputation + one-hot categorical...,StandardScaler on numeric columns only,LogisticRegression(max_iter=1000),Split 60/20/20 with threshold selection on val...,"max_iter=1000, random_state=42",42,0.3,0.7084,0.6273,0.9415,0.5000,0.5000,0.7530,0.5092,0.8185
2,P2,All week-4 features,Native categorical handling with explicit Miss...,NaN,CatBoostClassifier,Split 60/20/20 with threshold selection on val...,"loss_function=Logloss, eval_metric=AUC, depth=...",42,0.5,0.7414,0.6963,0.8018,0.6874,0.3126,0.7453,0.4989,0.8264
3,P2-T,All week-4 features,Native categorical handling with explicit Miss...,NaN,Calibrated CatBoostClassifier (sigmoid),Split 60/20/20 with threshold selection on val...,"loss_function=Logloss, eval_metric=AUC, depth=...",42,0.3,0.7294,0.6521,0.9145,0.5639,0.4361,0.7614,0.5044,0.8264


## 5. Short-Answer Word Counts

In [5]:
pd.read_csv(project_root / 'artifacts' / 'submission' / 'short_answer_word_counts.csv')

,answer,word_count
0,2.1,128
1,2.2,108
2,2.3,110
3,2.4,90
4,2.5,104
5,2.6,130
